# H07 — The PSD and whitening: a practical session

**Part VI · the laboratory of lecture 14, done in class**

**Data.**

- `labs/data/gw150914_H1.hdf5`: 32 s of strain of LIGO Hanford (H1) around GW150914, GPS 1126259447 to 1126259479, merger at 1126259462.42, sampled at 4096 Hz.
- `labs/data/gw150914_L1.hdf5`: the same 32 s in LIGO Livingston (L1), used in step 4 of Part B.
- Both files come from the Gravitational Wave Open Science Center (GWOSC), release R1. `load_strain` reads them and everything runs offline.
- The glitch of Part B is **simulated**: a model of what the instrument does, not a recording of it.

**Libraries.**

- `numpy`: arrays and the FFT.
- `scipy.signal`: `welch`, `lfilter`, `freqz`, `spectrogram` and the Tukey window.
- `matplotlib.pyplot`: all the plots.
- `h5py`: reads the data files, inside `gwcourse_signal`.
- `pycbc` and `lal`: the time series, the PSD, the waveform model of the template and the matched filter of A7.
- `gwcourse_signal`: the course functions that read and band-pass the strain.
- `tomllib`: reads `labs/config.toml`, the configuration of the laboratories: the folder of the data, the events and their files.
- `gwcourse_palette`: the course colors.

**What we do.**

- Part A, the demonstration: the data of lab H06 whitened, as an example; the data, the PSD with Welch's method, whitening in frequency and the crops, an AR model of the noise, whitening with the AR model, the event after whitening, the matched filter as a correlation of whitened data.
- Part B, on your own: change the order of the AR model, compare the two whitenings, put a glitch in the data and look at it before and after whitening, repeat on Livingston.

> **References**
>
> (Welch 1967), (Makhoul 1975), (LIGO Scientific Collaboration and Virgo Collaboration 2016)

## Tasks

**Part A. The demonstration.** We show how the plots of lecture 14 are made. All the code is written out and runs cell by cell. Under each cell we say what the result shows.

- **The example.** The data of lab H06, with its injected signal: band-passed as in H06, then whitened.
- **A1.** The data: the 32 s of H1 around GW150914, raw.
- **A2.** The PSD with Welch's method: the functions `estimate_psd` and `plot_psd`, two segment lengths.
- **A3.** Whitening in frequency: the functions `whiten_frequency` and `crop_edges`, the crops and the data and the PSD before and after.
- **A4.** An AR model of the noise: the functions `fit_ar` and `ar_psd`, three orders over the Welch estimate.
- **A5.** Whitening in time with the prediction error filter: the function `whiten_ar`, against the whitening in frequency.
- **A6.** The event after whitening and a band-pass.
- **A7.** The matched filter of lab H06 as a correlation of the whitened data with the whitened template.

**Part B. Your turn.** The functions of Part A are given, with three more for the glitch and the time–frequency map. We use them with our own choices.

- **B1.** Change the order of the AR model and see what happens to the lines.
- **B2.** Compare the whitening in frequency and the whitening in time.
- **B3.** Add a glitch to the data and make the time–frequency map before and after whitening.
- **B4.** Optional: the Livingston data or another segment length.

In [ ]:
# libraries
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# the course root holds gwcourse_palette.py: find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)
sys.path.insert(0, os.path.join(here, 'labs'))

# the configuration of the laboratories: where the data are and the events we use
import tomllib
with open(os.path.join(here, 'labs', 'config.toml'), 'rb') as config_file:
    config = tomllib.load(config_file)
DATA_DIR = os.path.join(here, 'labs', config['data']['dir'])
GPS_GW150914 = config['events']['GW150914']['gps']
H1_FILE = os.path.join(DATA_DIR, config['events']['GW150914']['H1'])
L1_FILE = os.path.join(DATA_DIR, config['events']['GW150914']['L1'])

# scipy.signal: Welch's method, the filter, its response, the spectrogram and the Tukey window
from scipy.signal import welch, lfilter, freqz, spectrogram
from scipy.signal.windows import tukey

# the course colors and the course functions that read and band-pass the strain
from gwcourse_palette import P, apply_style
from gwcourse_signal import load_strain, bandpass
apply_style()

# LAL redirects its printed output inside Jupyter and every waveform call becomes slow: switch it off
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

# the pycbc tools of A7: time series, filters, the PSD and the waveform models
from pycbc.types import TimeSeries
from pycbc.filter import matched_filter, highpass, resample_to_delta_t
from pycbc.filter import sigma as pycbc_sigma      # sigma is the name of the AR parameter in A4
from pycbc.psd import interpolate
from pycbc.waveform import get_td_waveform


## The example: the data of lab H06, whitened

In lab H06 we added a simulated signal to the Hanford data: a binary of 36 + 29 solar masses with SNR 20, 8 s after the start of the record. After a band-pass between 35 and 350 Hz the signal could not be seen in the data. Here we take the same data and the same signal and whiten them: we divide each frequency by the amplitude of the noise, $\sqrt{S_{n}(f)}$, so that the noise has the same level at every frequency. Sections A2 to A5 show how this is done, step by step.

The cell prepares the data as in H06, adds the signal and draws the same 3 s three times: the strain, the band-passed data and the whitened data. The band-passed and the whitened data are in units of their own standard deviation, measured on the data without the signal. In the whitened panel the signal alone, whitened in the same way, is drawn in brick.


In [ ]:
def prepare_h06_data(path, f_high_pass=15.0, fs_out=2048.0, edge=2.0, seglen=4.0):
    """Prepare the strain as in lab H06: high-pass, resampling, crop and the noise spectrum.

    Args:
        path: the file with the strain.
        f_high_pass: the frequency below which the strain is removed (Hz, default 15).
        fs_out: the sampling frequency after the resampling (Hz, default 2048).
        edge: the time cropped at each end (s, default 2).
        seglen: the length of the Welch segments (s, default 4).

    Returns:
        data: the prepared strain, a pycbc TimeSeries.
        psd: the one-sided PSD S_n(f) on the frequency grid of data.
    """
    # the strain of the file as a pycbc time series
    strain_h06, times_h06, fs_h06 = load_strain(path)
    series = TimeSeries(strain_h06, delta_t=1.0 / fs_h06, epoch=times_h06[0])
    # remove the slow drift, resample, crop the ends
    resampled = resample_to_delta_t(highpass(series, f_high_pass), 1.0 / fs_out)
    data = resampled.crop(edge, edge)
    # the Welch estimate of S_n on the frequency grid of the data
    psd = interpolate(data.psd(seglen), data.delta_f)
    return data, psd


def chirp_with_snr(like, psd, mass1, mass2, t_merger, rho=20.0, f_low=20.0):
    """Build the waveform of a binary with a chosen SNR, as long as the data.

    Args:
        like: the data, a pycbc TimeSeries: the waveform takes its sampling and length.
        psd: the one-sided PSD S_n(f) of the noise.
        mass1, mass2: the masses (solar masses).
        t_merger: the GPS time of the merger (s).
        rho: the SNR of the signal (default 20).
        f_low: the lower edge of the band (Hz, default 20).

    Returns:
        placed: the waveform in strain, with its merger at t_merger.
    """
    # the waveform at 1 Mpc and its SNR, which gives the factor for the SNR wanted
    wave, _ = get_td_waveform(approximant='IMRPhenomD', mass1=mass1, mass2=mass2,
                              delta_t=like.delta_t, f_lower=f_low)
    padded = wave.copy()
    padded.resize(len(like))
    rho_1 = pycbc_sigma(padded, psd=psd, low_frequency_cutoff=f_low)
    wave = wave * (rho / rho_1)
    # the merger at t_merger, in a series of zeros as long as the data
    wave.start_time = float(wave.start_time) + t_merger
    empty = TimeSeries(np.zeros(len(like)), delta_t=like.delta_t, epoch=like.start_time)
    placed = empty.add_into(wave)
    return placed


def whiten_series(series, psd, f_low=20.0, f_high=500.0):
    """Whiten a time series: divide its Fourier transform by sqrt(S_n) inside a band.

    Args:
        series: the data, a pycbc TimeSeries.
        psd: the one-sided PSD S_n(f) of the noise.
        f_low, f_high: the band that is kept (Hz, default 20 and 500).

    Returns:
        white: the whitened data, an array as long as series.
    """
    # the Fourier transform and the frequencies inside the band
    tilde = series.to_frequencyseries()
    freqs_band = tilde.sample_frequencies.numpy()
    band = (freqs_band >= f_low) & (freqs_band <= f_high)
    # divide by the amplitude of the noise, zero outside the band
    white_tilde = np.zeros(len(tilde), dtype=complex)
    white_tilde[band] = np.asarray(tilde)[band] / np.sqrt(np.asarray(psd)[band])
    white = np.fft.irfft(white_tilde, n=len(series))
    return white


# the data of H06 and the signal of H06: 36 + 29 solar masses, SNR 20, 8 s after the start
data_h06, psd_h06 = prepare_h06_data(H1_FILE)
t_inject = float(data_h06.start_time) + 8.0
signal_h06 = chirp_with_snr(data_h06, psd_h06, 36, 29, t_inject, rho=20.0)
with_signal = data_h06 + signal_h06
fs_h06 = 1.0 / data_h06.delta_t

# band-passed as in H06 and whitened, both in units of the standard deviation of the data alone
bp_alone = bandpass(np.asarray(data_h06), fs_h06, band=(35.0, 350.0))
bp_with = bandpass(np.asarray(with_signal), fs_h06, band=(35.0, 350.0))
wh_alone = whiten_series(data_h06, psd_h06)
wh_with = whiten_series(with_signal, psd_h06)
bp_unit = bp_alone.std()
wh_unit = wh_alone.std()

# the same 3 s, from 1.5 s before to 1.5 s after the merger of the signal
t_rel = data_h06.sample_times.numpy() - t_inject
look = np.abs(t_rel) < 1.5
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(t_rel[look], np.asarray(with_signal)[look], color=P['GREY_MID'], lw=0.5)
axes[0].set_ylabel('strain')
axes[0].set_title('the strain', loc='left')
axes[1].plot(t_rel[look], bp_with[look] / bp_unit, color=P['GREY_MID'], lw=0.5)
axes[1].set_ylabel('band-passed\n[std of the noise]')
axes[1].set_title('band-passed, 35 to 350 Hz, as in H06', loc='left')
axes[2].plot(t_rel[look], wh_with[look] / wh_unit, color=P['BLUE'], lw=0.5, label='data with the signal')
# the signal alone, whitened in the same way and drawn on top
wh_signal = whiten_series(signal_h06, psd_h06)
axes[2].plot(t_rel[look], wh_signal[look] / wh_unit, color=P['BRICK'], lw=1.2, label='the signal alone')
axes[2].legend(frameon=False, loc='lower right', ncol=2, bbox_to_anchor=(1.0, 1.0))
axes[2].set_ylabel('whitened\n[std of the noise]')
axes[2].set_title('whitened', loc='left')
axes[2].set_xlabel('time from the merger of the signal (s)')
plt.tight_layout()
plt.show()

# the largest value near the merger, and in the same stretch of data without the signal
near = np.abs(t_rel) < 0.05
print(f"band-passed: {np.abs(bp_with[near]).max() / bp_unit:.1f} near the merger, "
      f"data alone reach {np.abs(bp_alone[look]).max() / bp_unit:.1f} in these 3 s")
print(f"whitened:    {np.abs(wh_with[near]).max() / wh_unit:.1f} near the merger, "
      f"data alone reach {np.abs(wh_alone[look]).max() / wh_unit:.1f} in these 3 s")


**What the result shows.** In the strain and in the band-passed data the signal cannot be seen. Near the merger the band-passed data reach 2.5 standard deviations, less than the 2.9 of the same 3 s without the signal. After whitening the merger is a spike of 7.0 standard deviations, where the data alone reach 4.3. The brick curve shows that the spike is the signal: its last cycles, at the merger. The signal is the same, with SNR 20. Whitening gives every frequency the same level of noise, so the frequencies where the signal is strong stand out.


## A1. The data

Lecture 14 starts from the strain of the detector. We load the 32 s of H1 around GW150914 with `load_strain` and plot them as they are. The function `plot_series(times, x, ...)` draws a series against the time from the event. We plot all the 32 s, then half a second around the event.

In [ ]:
def plot_series(times, x, ax=None, t_event=GPS_GW150914, window=None, ylabel='strain', color=None, label=None):
    """Plot a time series against the time from the event.

    Args:
        times: the GPS time of each sample (s).
        x: the series, one value for each time.
        ax: the axes to draw on (default None: the current axes).
        t_event: the time that is drawn as zero (s, default the merger of GW150914).
        window: the stretch to show, (start, stop) in seconds from the event (default None: all of it).
        ylabel: the label of the vertical axis (default 'strain').
        color: the color of the line (default the dark grey of the course).
        label: the label of the line in a legend (default None).

    Returns:
        ax: the axes that were drawn on.
    """
    if ax is None:
        ax = plt.gca()
    if color is None:
        color = P['GREY_DARK']
    # the time from the event, and the samples inside the window
    t = times - t_event
    keep = np.ones(t.size, dtype=bool)
    if window is not None:
        keep = (t >= window[0]) & (t <= window[1])
    ax.plot(t[keep], x[keep], color=color, lw=0.8, label=label)
    ax.set_xlabel('time from the event (s)')
    ax.set_ylabel(ylabel)
    return ax


# the strain of H1: 32 s around GW150914, the GPS time of each sample and the sampling frequency
strain, times, fs = load_strain(H1_FILE)
dt = 1.0 / fs
print(f"{strain.size} samples at {fs:.0f} Hz = {strain.size / fs:.0f} s, dt = {dt * 1e3:.3f} ms")
print(f"root mean square strain {strain.std():.2e}")

# the whole record, then half a second around the event
fig, axes = plt.subplots(2, 1, figsize=(10, 5))
plot_series(times, strain, ax=axes[0])
axes[0].set_title('all 32 s', loc='left')
plot_series(times, strain, ax=axes[1], window=(-0.25, 0.25))
axes[1].axvline(0.0, color=P['BRICK'], lw=1.5, ls='--')
axes[1].set_title('half a second around the event (dashed: the merger)', loc='left')
plt.tight_layout()
plt.show()

**What the result shows.** The record has 131072 samples at 4096 Hz, so 32 s and $\Delta t = 0.244$ ms. The root mean square strain is $2.18\times10^{-19}$. The record is a slow wave of order $10^{-19}$, the noise at low frequency. The half second around the event looks the same. There is no feature at the merger: the signal is much smaller than this wave.

## A2. How to estimate the PSD

Lecture 14, the PSD: Welch averages the periodograms of segments of $L$ samples. It resolves no detail narrower than about $1/(L\Delta t)$.

`estimate_psd(x, fs, seglen=4.0)` is Welch's method. It returns the frequencies and $S_{n} = 2S_{xx}$, the one-sided PSD in 1/Hz for strain, with $f$ in hertz. `plot_psd(freqs, psd, ...)` plots $\sqrt{S_{n}}$, the amplitude spectral density, on log axes. We use segments of 4 s and then segments of 1 s.

In [ ]:
def plot_psd(freqs, psd, ax=None, label=None, color=None, lw=1.0, xlim=(10, 2000)):
    """Plot the amplitude spectral density, the square root of S_n, on log axes.

    Args:
        freqs: the frequencies of the PSD (Hz).
        psd: the one-sided PSD S_n at those frequencies (1/Hz for strain).
        ax: the axes to draw on (default None: the current axes).
        label: the label of the curve in a legend (default None).
        color: the color of the curve (default the dark grey of the course).
        lw: the width of the line (default 1.0).
        xlim: the range of frequencies to show (Hz, default 10 to 2000).

    Returns:
        ax: the axes that were drawn on.
    """
    if ax is None:
        ax = plt.gca()
    if color is None:
        color = P['GREY_DARK']
    # the amplitude spectral density; the zero frequency cannot go on a log axis
    asd = np.sqrt(psd[1:])
    ax.loglog(freqs[1:], asd, color=color, lw=lw, label=label)
    ax.set_xlim(*xlim)
    ax.set_xlabel('frequency (Hz)')
    ax.set_ylabel(r'$\sqrt{S_{n}(f)}$  (Hz$^{-1/2}$)')
    return ax


def estimate_psd(x, fs, seglen=4.0):
    """Estimate the one-sided PSD S_n = 2 S_xx with Welch's method.

    Args:
        x: the data, a one-dimensional array.
        fs: the sampling frequency (Hz).
        seglen: the length of the segments that are averaged (s, default 4).

    Returns:
        freqs: the frequencies (Hz), spaced by 1 / seglen.
        psd: the one-sided PSD S_n at those frequencies (1/Hz for strain).
    """
    # segments of seglen seconds, a Hann window on each, half of each segment overlaps the next
    freqs, psd = welch(x, fs=fs, nperseg=int(seglen * fs), window='hann')
    return freqs, psd


# Welch's method: S_n with segments of 4 s and with segments of 1 s
freqs, psd = estimate_psd(strain, fs=fs, seglen=4.0)
freqs_1, psd_1 = estimate_psd(strain, fs=fs, seglen=1.0)

# for each: the frequency step, the number of segments, the scatter of the continuum
# between 200 and 290 Hz and the height of the 60 Hz line (largest ASD within 2 Hz of it)
for seglen, f_w, s_w in ((4.0, freqs, psd), (1.0, freqs_1, psd_1)):
    n_segment = (strain.size - int(seglen * fs)) // (int(seglen * fs) // 2) + 1
    continuum = s_w[(f_w > 200) & (f_w < 290)]
    near_60 = (f_w > 58) & (f_w < 62)
    print(f"{seglen:.0f} s segments: step {f_w[1]:.2f} Hz, {n_segment} segments, "
          f"scatter {100 * continuum.std() / continuum.mean():.0f} %, "
          f"60 Hz line {np.sqrt(s_w[near_60].max()):.1e}")

# the amplitude spectral density sqrt(S_n) of both
fig, ax = plt.subplots(figsize=(10, 5))
plot_psd(freqs_1, psd_1, ax=ax, color=P['AMBER'], label='1 s segments')
plot_psd(freqs, psd, ax=ax, color=P['BLUE'], label='4 s segments')
ax.set_ylim(1e-24, 1e-18)
ax.legend(frameon=False, loc='upper right')
plt.show()

**What the result shows.** The two curves have the same shape: the wall below 30 Hz, the bucket between 100 and 300 Hz and the lines above it, at 36 Hz, 60 Hz and its harmonics, 330 Hz, 500 Hz, 1000 Hz and 1500 Hz.

| segments | step $1/L\Delta t$ | number | scatter of the continuum | height of the 60 Hz line |
|---|---|---|---|---|
| 4 s | 0.25 Hz | 15 | 28 % | $6.9\times10^{-22}$ |
| 1 s | 1.00 Hz | 63 | 16 % | $3.5\times10^{-22}$ |

Long segments resolve the lines and give a larger scatter. Short segments average more and give a smoother curve, but a narrow line is wider and lower. The data are 32 s long: we cannot have both. We use 4 s in the rest of the laboratory.

## A3. How to whiten

Lecture 14, whitening in frequency: we divide the transform $D_{k}$ of the data by $\sqrt{S_{n}(f_{k})/(2\Delta t)}$, so that the whitened noise has unit variance in each sample. The steps are six: the PSD with Welch, a window on the data, the FFT, the division, the inverse FFT and the crop of the edges.

`whiten_frequency(x, fs, spectrum=None, alpha=0.125)` does the middle four steps, with the PSD of A2. `crop_edges(x, fs, start=2.0, stop=2.0)` does the last one.

### Whitening in frequency

In [ ]:
def whiten_frequency(x, fs, spectrum=None, alpha=0.125):
    """Whiten data in frequency: divide the transform by the amplitude of the noise.

    The edges of the output are corrupted: drop them with crop_edges.

    Args:
        x: the data, a one-dimensional array.
        fs: the sampling frequency (Hz).
        spectrum: the pair (freqs, S_n) from estimate_psd (default None: S_n is estimated from x with 4 s segments).
        alpha: the fraction of the Tukey window that is taper (default 0.125).

    Returns:
        whitened: the whitened data, as long as x, with unit variance away from the edges.
    """
    if spectrum is None:
        spectrum = estimate_psd(x, fs=fs, seglen=4.0)
    freqs_psd, s_n = spectrum
    # the window takes the two ends of the data smoothly to zero, then the FFT
    n = x.size
    d_k = np.fft.rfft(x * tukey(n, alpha=alpha))
    # S_n / (2 dt) is the variance of one sample of white noise with this PSD
    freqs_dft = np.fft.rfftfreq(n, 1.0 / fs)
    s_n_dft = np.interp(freqs_dft, freqs_psd, s_n)
    amplitude = np.sqrt(s_n_dft * fs / 2)
    amplitude[amplitude <= 0] = np.inf
    # divide each frequency by the amplitude of the noise, then back to time
    whitened = np.fft.irfft(d_k / amplitude, n=n)
    return whitened


# whiten in frequency: the transform is divided by the amplitude of the noise, from the 4 s estimate above
white_full = whiten_frequency(strain, fs=fs, spectrum=(freqs, psd))

# the whitened data with all its samples; the first and the last 2 s are shaded
t_start, t_stop = times[0] - GPS_GW150914, times[-1] - GPS_GW150914
fig, ax = plt.subplots(figsize=(10, 3.5))
plot_series(times, white_full, ax=ax, ylabel='whitened strain')
ax.axvspan(t_start, t_start + 2.0, color=P['BRICK'], alpha=0.3)
ax.axvspan(t_stop - 2.0, t_stop, color=P['BRICK'], alpha=0.3)
plt.tight_layout()
plt.show()

# the standard deviation in blocks of 0.5 s, from the start of the record and then in the middle
half_s = int(0.5 * fs)
for k in range(5):
    block = white_full[k * half_s:(k + 1) * half_s]
    print(f"{0.5 * k:.1f}-{0.5 * (k + 1):.1f} s from the start: {block.std():.2f}")
print(f"middle of the record (8 to 24 s): {white_full[int(8 * fs):int(24 * fs)].std():.2f}")

**What the result shows.** Away from the edges the whitened data have a constant level. The two shaded stretches are not valid. The standard deviation in blocks of 0.5 s is 0.11, 0.34, 0.69, 0.94 and then 1.00. In the middle of the record it is 0.98. The window takes the data to zero at the two ends and the division spreads that over the first and the last 2 s. These samples are corrupted.

### The crops

`crop_edges` removes the first and the last 2 s, from the whitened data and from the times that go with them.

In [ ]:
def crop_edges(x, fs, start=2.0, stop=2.0):
    """Remove the first and the last seconds of a series.

    Args:
        x: the series, a one-dimensional array (the whitened data, or the times that go with it).
        fs: the sampling frequency (Hz).
        start: the time removed at the start (s, default 2).
        stop: the time removed at the end (s, default 2).

    Returns:
        cropped: the series without its edges.
    """
    # the number of samples at each end; the end is counted from the last sample
    n_start = int(round(start * fs))
    n_stop = int(round(stop * fs))
    cropped = x[n_start:x.size - n_stop]
    return cropped


# drop the first and the last 2 s, from the whitened data and from the times
white = crop_edges(white_full, fs=fs, start=2.0, stop=2.0)
t_white = crop_edges(times, fs=fs, start=2.0, stop=2.0)
print(f"{white.size / fs:.0f} s are left, from {t_white[0] - GPS_GW150914:.2f} to {t_white[-1] - GPS_GW150914:.2f} s from the event")
print(f"standard deviation with the edges {white_full.std():.3f}, without {white.std():.3f}")

# the whitened data without the edges
fig, ax = plt.subplots(figsize=(10, 3.5))
plot_series(t_white, white, ax=ax, ylabel='whitened strain', color=P['BLUE'])
plt.tight_layout()
plt.show()

**What the result shows.** 28 s are left, from $-13.42$ to $14.58$ s from the event. The standard deviation is 0.945 with the edges and 0.984 without them. From here on `white` and `t_white` are the whitened data and their times.

### Before and after whitening, in time

A quarter of a second, 10 s before the merger, far from the event and from the edges. The raw strain and the whitened data are drawn on the same axes, each divided by its own standard deviation.


In [ ]:
# two seconds, from 10 to 8 s before the merger, far from the event and from the edges
raw_stretch = (times - GPS_GW150914 > -10) & (times - GPS_GW150914 < -8)
white_stretch = (t_white - GPS_GW150914 > -10) & (t_white - GPS_GW150914 < -8)
raw_std = strain[raw_stretch].std()
white_std = white[white_stretch].std()

# a quarter of a second on the same axes, each in units of its own standard deviation; the raw strain on top
fig, ax = plt.subplots(figsize=(10, 3.8))
plot_series(t_white, white / white_std, ax=ax, window=(-10, -9.75), color=P['BLUE'], label='whitened')
plot_series(times, strain / raw_std, ax=ax, window=(-10, -9.75), color=P['BRICK'], label='raw strain')
ax.lines[-1].set_linewidth(2.0)
ax.set_ylabel('in units of its standard deviation')
ax.legend(frameon=False, loc='lower right', ncol=2, bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

# the standard deviation of each
print(f"raw: {raw_std:.2e}, whitened: {white_std:.2f}")


**What the result shows.** With the same scale the difference is clear. The raw strain, standard deviation $2.76\times10^{-19}$, is a slow oscillation: the noise below 30 Hz dominates it. The whitened data, standard deviation 1.00, change from one sample to the next as white noise does: every frequency has the same weight.


### Before and after whitening, in frequency

The PSD of the whitened data, with the same method and the same segments. We normalize the two spectra so that they fit on the same plot.

In [ ]:
# the PSD of the whitened data, with the same method and the same segments
freqs_w, psd_w = estimate_psd(white, fs=fs, seglen=4.0)

# the two PSDs differ by many orders of magnitude, so we normalize each one:
# we divide it by its median between 100 and 300 Hz, where the detector is most sensitive
band_ref = (freqs >= 100) & (freqs < 300)
band_ref_w = (freqs_w >= 100) & (freqs_w < 300)
psd_norm = psd / np.median(psd[band_ref])
psd_w_norm = psd_w / np.median(psd_w[band_ref_w])

# before and after on the same plot: after whitening the spectrum is flat
plt.figure(figsize=(10, 4.6))
plot_psd(freqs, psd_norm, color=P['BRICK'], label='before: the raw strain')
plot_psd(freqs_w, psd_w_norm, color=P['BLUE'], label='after: whitened')
plt.ylim(3e-2, 3e4)
plt.ylabel(r'$\sqrt{S_{n}(f)}$, divided by its median at 100-300 Hz')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# the level before and after, in four bands: the median of S_n; after, in units of 2 dt
print(f"unit variance: 2 dt = {2 * dt:.2e} 1/Hz, sqrt(2 dt) = {np.sqrt(2 * dt):.3f}")
for low, high in ((20, 100), (100, 300), (300, 1000), (1000, 1800)):
    band = (freqs >= low) & (freqs < high)
    band_w = (freqs_w >= low) & (freqs_w < high)
    print(f"{low:4d}-{high:4d} Hz: before {np.sqrt(np.median(psd[band])):.1e}, "
          f"after {np.median(psd_w[band_w]) / (2 * dt):.2f} (units of 2 dt)")

**What the result shows.** The two spectra are on the same plot, each divided by its median between 100 and 300 Hz. Before whitening the spectrum changes by several orders of magnitude between 10 Hz and 2 kHz. After whitening it is flat. Whitened data of unit variance have a flat one-sided PSD at $2\Delta t = 4.88\times10^{-4}$ 1/Hz. The median of $S_{n}$ in units of $2\Delta t$ is 0.98, 0.99, 0.98 and 0.99 in the four bands, while before whitening the median of $\sqrt{S_{n}}$ goes from $1.5\times10^{-23}$ at 20–100 Hz to $8.1\times10^{-24}$ at 100–300 Hz and $2.3\times10^{-23}$ at 1000–1800 Hz. The lines are gone from the spectrum. Narrow notches remain at their frequencies. Welch widens a narrow line, as in lecture 14, so the division removes a little more than the line.

## A4. How to estimate an AR model

Lecture 14, the autoregressive model: $x[n] = -\sum_{k=1}^{p} a_{k}\,x[n-k] + w[n]$, with $w$ a white noise of standard deviation $\sigma$. The model has the PSD $P_{\mathrm{AR}}(f) = \sigma^{2}\Delta t/|A(f)|^{2}$. In the one-sided form it is $S_{n} = 2P_{\mathrm{AR}}$. `fit_ar(x, order=512)` finds the coefficients $a_{k}$ and $\sigma$ with Burg's method, which uses no window and no estimate of $r_{xx}$. `ar_psd(a, sigma, fs, freqs)` gives the spectrum of the model. We fit three orders and plot them over the Welch estimate.

In [ ]:
def fit_ar(x, order=512):
    """Fit an AR model to the data with Burg's method.

    The model is x[n] = -sum_k a_k x[n-k] + w[n], with w a white noise of standard deviation sigma.

    Args:
        x: the data, a one-dimensional array.
        order: the order p of the model, the number of coefficients (default 512).

    Returns:
        a: the coefficients a_1 ... a_p, an array of p values.
        sigma: the standard deviation of the driving noise w, in the units of x.
    """
    # Burg's method uses no window and no autocorrelation: only the mean is taken out
    x = x - x.mean()
    # order 0: both prediction errors are the data, the backward one a sample earlier
    e_f = x[1:]
    e_b = x[:-1]
    a = np.zeros(order)
    variance = np.mean(x ** 2)
    for m in range(1, order + 1):
        # the reflection coefficient that minimizes the forward plus the backward error
        k = -2.0 * np.sum(e_f * e_b) / (np.sum(e_f ** 2) + np.sum(e_b ** 2))
        # the coefficients of order m from those of order m - 1, and the new variance
        a_old = a[:m - 1].copy()
        a[:m - 1] = a_old + k * a_old[::-1]
        a[m - 1] = k
        variance = variance * (1.0 - k ** 2)
        # the errors of order m; the backward one is delayed by one sample for the next order
        new_f = e_f + k * e_b
        new_b = e_b + k * e_f
        e_f = new_f[1:]
        e_b = new_b[:-1]
    sigma = np.sqrt(variance)
    return a, sigma


def ar_psd(a, sigma, fs, freqs):
    """Compute the one-sided PSD S_n = 2 P_AR of an AR model.

    P_AR = sigma^2 dt / |A(f)|^2, with A(f) = 1 + sum_k a_k exp(-i 2 pi f k dt).

    Args:
        a: the coefficients a_1 ... a_p of the model, from fit_ar.
        sigma: the standard deviation of the driving noise, from fit_ar.
        fs: the sampling frequency (Hz).
        freqs: the frequencies where S_n is computed (Hz).

    Returns:
        psd: the one-sided PSD S_n of the model at freqs.
    """
    # A(f) on the frequencies asked for, from the polynomial 1 + a_1 z^-1 + ... + a_p z^-p
    _, a_f = freqz(np.r_[1.0, a], worN=freqs, fs=fs)
    psd = 2.0 * sigma ** 2 / fs / np.abs(a_f) ** 2
    return psd


# AR models of three orders, fitted with Burg's method on the 32 s of H1
orders = (32, 128, 512)
models = {}
for order in orders:
    a, sigma = fit_ar(strain, order=order)
    models[order] = (a, sigma)
    print(f"order {order:4d}: sigma = {sigma:.2e}")

# the spectra of the models on the frequencies of the Welch estimate, over the Welch curve
colors = (P['AMBER'], P['BLUE'], P['BRICK'])
fig, ax = plt.subplots(figsize=(10, 5))
plot_psd(freqs, psd, ax=ax, color=P['GREY_MID'], lw=2.0, label='Welch, 4 s segments')
for order, color in zip(orders, colors):
    a, sigma = models[order]
    psd_ar = ar_psd(a, sigma, fs=fs, freqs=freqs)
    plot_psd(freqs, psd_ar, ax=ax, color=color, label=f'AR({order})')
    # the largest ASD within 2 Hz of the mains line and of the violin mode
    for f_line in (60.0, 501.8):
        near = np.abs(freqs - f_line) < 2
        print(f"  AR({order}) at {f_line:5.1f} Hz: {np.sqrt(psd_ar[near].max()):.1e}")
ax.set_ylim(1e-24, 1e-18)
ax.legend(frameon=False, loc='upper right')
plt.show()

# the same lines in the Welch estimate
for f_line in (60.0, 501.8):
    near = np.abs(freqs - f_line) < 2
    print(f"Welch at {f_line:5.1f} Hz: {np.sqrt(psd[near].max()):.1e}")

**What the result shows.** The spectrum of the model is smooth and the lines appear as the order $p$ grows. $\sigma$ is $8.56\times10^{-22}$, $6.97\times10^{-22}$ and $6.15\times10^{-22}$ for $p = 32$, 128 and 512. AR(32) gives the slope and no lines. AR(128) adds a bump near 60 Hz and the lines at 500 Hz and 1000 Hz, too wide. AR(512) has a narrow peak on the strong lines.

| ASD at | Welch | AR(32) | AR(128) | AR(512) |
|---|---|---|---|---|
| 60 Hz | $6.9\times10^{-22}$ | $8.4\times10^{-23}$ | $1.4\times10^{-22}$ | $1.5\times10^{-21}$ |
| 501.8 Hz | $8.9\times10^{-22}$ | $2.7\times10^{-22}$ | $8.0\times10^{-22}$ | $1.1\times10^{-21}$ |

At 60 Hz the AR(512) peak is twice the Welch one. The width of a line in the model comes from the radius of its poles, not from the length of a segment.

## A5. How to use the AR model for whitening

Lecture 14, the prediction error filter: the data through $A(z)$ give $e[n] = x[n] + \sum_{k=1}^{p} a_{k}\,x[n-k]$, a white series with variance $\sigma^{2}$. `whiten_ar(x, a, sigma)` is one call, with the AR(512) model of A4. The filter uses only the past, so the first $p$ samples have no past.

In [ ]:
def whiten_ar(x, a, sigma):
    """Whiten data in time with the prediction error filter A(z) of an AR model.

    The first p samples have no past: drop them with crop_edges.

    Args:
        x: the data, a one-dimensional array.
        a: the coefficients a_1 ... a_p of the model, from fit_ar.
        sigma: the standard deviation of the driving noise, from fit_ar.

    Returns:
        whitened: the prediction error divided by sigma, as long as x, with unit variance.
    """
    # e[n] = x[n] + sum_k a_k x[n-k]: the data through A(z), which uses the past only
    prediction_error = lfilter(np.r_[1.0, a], 1.0, x)
    whitened = prediction_error / sigma
    return whitened


# the AR(512) model: the data through the prediction error filter A(z), in one call
order = 512
a, sigma = models[order]
white_ar_full = whiten_ar(strain, a=a, sigma=sigma)

# the first 512 samples have no past: how large are they, against the rest?
print(f"largest value in the first {order} samples: {np.abs(white_ar_full[:order]).max():.1f}")
print(f"largest value in the rest: {np.abs(white_ar_full[order:]).max():.1f}")

# drop them, and the times that go with them; nothing is dropped at the end
white_ar = crop_edges(white_ar_full, fs=fs, start=order / fs, stop=0.0)
t_white_ar = crop_edges(times, fs=fs, start=order / fs, stop=0.0)
print(f"{order / fs * 1e3:.0f} ms dropped; standard deviation {white_ar.std():.3f}")

# the first 0.5 s of the record, with the samples that are dropped shaded
fig, ax = plt.subplots(figsize=(10, 3.2))
plot_series(times, white_ar_full, ax=ax, t_event=times[0], window=(0.0, 0.5), ylabel='whitened strain')
ax.axvspan(0.0, order / fs, color=P['BRICK'], alpha=0.3)
ax.set_xlabel('time from the start of the record (s)')
plt.tight_layout()
plt.show()

**What the result shows.** The largest value in the first 512 samples is 59.3. In the rest of the record it is 5.1. We drop the 512 samples (125 ms) and nothing at the end. The standard deviation of the rest is 1.000.

### The two whitenings

The same two seconds and the PSD of the data, whitened in frequency with the Welch PSD and in time with AR(512).

In [ ]:
# the PSD of the data whitened in time, with the same method as before
freqs_ar, psd_ar_white = estimate_psd(white_ar, fs=fs, seglen=4.0)

# left: the same 2 s in both whitenings; right: the PSD of both
fig = plt.figure(figsize=(11, 5))
ax1 = fig.add_subplot(2, 2, 1)
plot_series(t_white, white, ax=ax1, window=(-10, -8), color=P['BLUE'], ylabel='whitened strain')
ax1.set_title('in frequency', loc='left')
ax2 = fig.add_subplot(2, 2, 3)
plot_series(t_white_ar, white_ar, ax=ax2, window=(-10, -8), color=P['BRICK'], ylabel='whitened strain')
ax2.set_title('in time, AR(512)', loc='left')
ax3 = fig.add_subplot(1, 2, 2)
plot_psd(freqs_w, psd_w, ax=ax3, color=P['BLUE'], label='in frequency')
plot_psd(freqs_ar, psd_ar_white, ax=ax3, color=P['BRICK'], label='in time, AR(512)')
ax3.axhline(np.sqrt(2 * dt), color=P['GREY_MID'], lw=1.2, ls='--')
ax3.set_ylim(1e-3, 1e0)
ax3.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

# for each whitening: the standard deviation and the largest S_n between 20 and 1800 Hz, in units of 2 dt
for name, w, f_w, s_w in (('in frequency', white, freqs_w, psd_w), ('in time', white_ar, freqs_ar, psd_ar_white)):
    band = (f_w > 20) & (f_w < 1800)
    i_max = np.argmax(s_w[band])
    print(f"{name:13s}: standard deviation {w.std():.3f}, "
          f"largest S_n {s_w[band][i_max] / (2 * dt):.1f} at {f_w[band][i_max]:.1f} Hz")

# a larger order: AR(2048), whitened in time in the same way
a_2048, sigma_2048 = fit_ar(strain, order=2048)
white_2048 = whiten_ar(strain, a=a_2048, sigma=sigma_2048)[2048:]
freqs_2048, psd_2048 = estimate_psd(white_2048, fs=fs, seglen=4.0)
band = (freqs_2048 > 20) & (freqs_2048 < 1800)
print(f"AR(2048)     : largest S_n {psd_2048[band].max() / (2 * dt):.1f} at "
      f"{freqs_2048[band][np.argmax(psd_2048[band])]:.1f} Hz")

**What the result shows.** Both series look like white noise. In both, the PSD is flat at $2\Delta t$ over most of the band. The whitening in frequency has a standard deviation of 0.984 and notches at the lines. The whitening in time has 1.000. The largest value of $S_{n}$ between 20 Hz and 1800 Hz is 1.4 times $2\Delta t$ in frequency, at 1169.0 Hz. It is 20.2 times $2\Delta t$ in time, at 1468.0 Hz, in the group of lines near 1.5 kHz: the model with $p = 512$ does not follow every line.

**Why whitening in time, if whitening in frequency is better here?**

- **The lines.** The narrow lines of the detector, such as the group near 1.5 kHz, are narrower than what an AR model of this order can follow. A larger order does not remove them: AR(2048) still leaves a peak of 16.9 times $2\Delta t$. In practice the lines are removed first, with notch filters or with a subtraction, and the AR model whitens the broad noise.
- **Time.** The prediction error filter uses only the past. It whitens the data sample by sample, as they arrive, with a delay of one sample. Whitening in frequency needs the whole stretch at once and loses its two edges.
- **A noise that changes.** The coefficients of the model can be fitted again on the newest data, so the filter follows a noise that drifts. A search that works on the data stream, in low latency, uses a filter of this kind.


## A6. The event after whitening

Lecture 14, GW150914 in Hanford: raw, whitened and band-passed. We take the whitened data of A3 and keep the band from 35 to 350 Hz with `bandpass`, forwards and backwards, so that the phase is not changed. $\sigma_{x}$ is the standard deviation of the band-passed data more than 1 s from the event.

In [ ]:
# band-pass the whitened data between 35 and 350 Hz, forwards and backwards, so the phase is not changed
filtered = bandpass(white, fs=fs, band=(35.0, 350.0))

# the units of the noise: the standard deviation away from the event (more than 1 s from it)
quiet = np.abs(t_white - GPS_GW150914) > 1.0
sigma_x = filtered[quiet].std()
near = np.abs(t_white - GPS_GW150914) < 0.2
i_peak = np.argmax(np.abs(filtered[near]))
print(f"sigma_x of the band-passed data: {sigma_x:.2f}")
print(f"largest value within 0.2 s of the event: {np.abs(filtered[near][i_peak]) / sigma_x:.1f} sigma_x, "
      f"{1e3 * (t_white[near][i_peak] - GPS_GW150914):+.0f} ms from the event")

# 0.4 s around the event: the whitened data, then the same band-passed
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
plot_series(t_white, white, ax=axes[0], window=(-0.3, 0.1), color=P['BLUE'], ylabel='whitened strain')
axes[0].set_title('whitened', loc='left')
plot_series(t_white, filtered / sigma_x, ax=axes[1], window=(-0.3, 0.1), color=P['BRICK'], ylabel=r'band-passed, in $\sigma_{x}$')
axes[1].axvline(0.0, color=P['GREY_MID'], lw=1.2, ls='--')
axes[1].set_title('35 to 350 Hz (dashed: the merger)', loc='left')
plt.tight_layout()
plt.show()

**What the result shows.** $\sigma_{x} = 0.36$. The whitened data do not show the event. After the band-pass the signal is clear: a few cycles that rise in frequency and in amplitude and end at the merger. The largest value within 0.2 s of the event is $8.1\,\sigma_{x}$, at 3 ms from the event time. This is the chirp of GW150914 in Hanford.

## A7. The matched filter as a correlation of whitened data

The matched filter of lecture 13 weights the inner product with $1/S_{n}$. Lecture 14 splits this weight into two equal factors, one on each function:

$$
\langle s,h\rangle = 4\,\Re\int_{0}^{\infty}\mathrm{d}f\;
\tilde{s}_{w}(f)\,\tilde{h}_{w}^{*}(f),
\qquad \tilde{s}_{w} = \frac{\tilde{s}}{\sqrt{S_{n}}},
\quad \tilde{h}_{w} = \frac{\tilde{h}}{\sqrt{S_{n}}}
$$

So the matched filter whitens the data and the template and then correlates them. We check this on the Hanford data with the template of lab H06: 36 + 29 solar masses, waveform model `IMRPhenomD`, at 1 Mpc.

- `condition_for_filter` prepares the data as in H06: high-pass at 15 Hz, resampling to 2048 Hz, 2 s cropped at each end, $S_{n}$ with Welch's method on 4 s segments.
- `make_template` builds the template, as long as the data, with the merger at time zero.
- `phi_from_whitened` divides the Fourier transforms of the data and of the template by $\sqrt{S_{n}}$ above 20 Hz. It computes $\langle h,h\rangle = 4\Delta f\sum|\tilde{h}_{w}|^{2}$, with $\Delta f$ the frequency step, and $\varrho_{h}$, its square root. It computes $\Phi(t_{c})$ with one inverse FFT of $\tilde{s}_{w}\tilde{h}_{w}^{*}$, divided by $\varrho_{h}$, and takes the modulus of the complex output, as in lecture 13.

Then we compare $\varrho_{h}$ with the function `sigma` of `pycbc` and $\Phi(t_{c})$ with the filter of `pycbc`. The plot draws the two outputs around GW150914 on the same axes, with the numbers in a box.


In [ ]:
def condition_for_filter(strain, times, fs, f_high_pass=15.0, fs_out=2048.0, edge=2.0, seglen=4.0):
    """Prepare the strain for the matched filter of pycbc, as in lab H06.

    Args:
        strain: the strain, a one-dimensional array.
        times: the GPS time of each sample (s).
        fs: the sampling frequency of the strain (Hz).
        f_high_pass: the frequency below which the strain is removed (Hz, default 15).
        fs_out: the sampling frequency after the resampling (Hz, default 2048).
        edge: the time cropped at each end (s, default 2).
        seglen: the length of the Welch segments (s, default 4).

    Returns:
        data: the strain, high-passed, resampled and cropped, a pycbc TimeSeries.
        psd: the one-sided PSD S_n(f) on the frequency grid of data.
    """
    # the strain as a pycbc time series, without the slow drift, resampled
    series = TimeSeries(strain, delta_t=1.0 / fs, epoch=times[0])
    filtered = highpass(series, f_high_pass)
    resampled = resample_to_delta_t(filtered, 1.0 / fs_out)
    # crop the ends, then the Welch estimate on the frequency grid of the data
    data = resampled.crop(edge, edge)
    psd = interpolate(data.psd(seglen), data.delta_f)
    return data, psd


def make_template(mass1, mass2, like, f_low=20.0, approximant='IMRPhenomD'):
    """Build the template of a binary at 1 Mpc, as long as the data.

    Args:
        mass1, mass2: the masses (solar masses).
        like: the time series whose sampling and length the template takes.
        f_low: the lower edge of the band (Hz, default 20).
        approximant: the waveform model (default 'IMRPhenomD').

    Returns:
        shifted: the template, with the merger at time zero.
    """
    # the waveform at 1 Mpc, made as long as the data
    hp, _ = get_td_waveform(approximant=approximant, mass1=mass1, mass2=mass2,
                            delta_t=like.delta_t, f_lower=f_low)
    hp.resize(len(like))
    # the merger at time zero
    shifted = hp.cyclic_time_shift(hp.start_time)
    return shifted


def phi_from_whitened(series, template, psd, f_low=20.0):
    """Compute Phi(t_c) and rho_h from the whitened data and the whitened template.

    Args:
        series: the data, a pycbc TimeSeries.
        template: the template, as long as the data.
        psd: the one-sided PSD S_n(f) of the noise.
        f_low: the lower edge of the band (Hz, default 20).

    Returns:
        phi_w: |Phi(t_c)|, an array as long as the data.
        rho_h_w: rho_h = sqrt(<h,h>), the SNR of a signal equal to the template.
    """
    # the Fourier transforms and the frequencies above f_low
    s_tilde = series.to_frequencyseries()
    h_tilde = template.to_frequencyseries()
    freqs = s_tilde.sample_frequencies.numpy()
    band = freqs >= f_low
    # divide the transform of each by the amplitude of the noise, sqrt(S_n)
    amplitude = np.sqrt(np.asarray(psd)[band])
    s_w = np.asarray(s_tilde)[band] / amplitude
    h_w = np.asarray(h_tilde)[band] / amplitude
    # <h,h> = 4 df sum |h_w|^2, and rho_h is its square root
    delta_f = series.delta_f
    rho_h_w = np.sqrt(4 * delta_f * np.sum(np.abs(h_w) ** 2))
    # <s,h>(t_c) = 4 df sum s_w h_w^* exp(2 pi i f t): one inverse FFT, zero below f_low
    n = len(series)
    product = np.zeros(n // 2 + 1, dtype=complex)
    product[band] = s_w * np.conj(h_w)
    inner = 4 * delta_f * n * np.fft.ifft(np.r_[product, np.zeros(n - product.size)])
    phi_w = np.abs(inner) / rho_h_w
    return phi_w, rho_h_w


# the Hanford data prepared for the filter and the template of lab H06
data, psd_filter = condition_for_filter(strain, times, fs)
template = make_template(36, 29, like=data)

# Phi from the whitened data and template, and from the filter of pycbc
phi_w, rho_h_w = phi_from_whitened(data, template, psd_filter)
rho_h = pycbc_sigma(template, psd=psd_filter, low_frequency_cutoff=20)
reference = matched_filter(template, data, psd=psd_filter, low_frequency_cutoff=20)
print(f"rho_h: {rho_h_w:.1f} from the whitened template, {rho_h:.1f} from sigma")
print(f"largest difference between the two Phi(t_c): "
      f"{np.max(np.abs(phi_w - np.abs(np.asarray(reference)))):.1e}")

# the output without 6 s at the start and 4 s at the end, and the value at the event
phi_series = TimeSeries(phi_w, delta_t=data.delta_t, epoch=data.start_time).crop(6, 4)
phi_reference = reference.crop(6, 4)
t_axis = phi_series.sample_times.numpy()
near = np.abs(t_axis - GPS_GW150914) < 0.1
print(f"Phi at the event: {np.asarray(phi_series)[near].max():.2f} whitened, "
      f"{np.abs(np.asarray(phi_reference))[near].max():.2f} matched_filter")

# the two outputs around the event, on the same axes, with the numbers in a box
t_ms = 1e3 * (t_axis - GPS_GW150914)
window = np.abs(t_ms) < 100
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_ms[window], np.abs(np.asarray(phi_reference))[window], color=P['GREY_MID'], lw=4.0,
        label='filter of pycbc')
ax.plot(t_ms[window], np.asarray(phi_series)[window], color=P['BRICK'], lw=1.2, ls='--',
        label='whitened data times whitened template')
ax.set_xlabel('time from GW150914 (ms)')
ax.set_ylabel(r'$|\Phi(t_c)|$')
ax.legend(frameon=False, loc='upper left')
summary = (f"rho_h: {rho_h_w:.1f} and {rho_h:.1f}\n"
           f"largest difference of Phi: {np.max(np.abs(phi_w - np.abs(np.asarray(reference)))):.0e}\n"
           f"Phi at the event: {np.asarray(phi_series)[near].max():.2f} and "
           f"{np.abs(np.asarray(phi_reference))[near].max():.2f}")
ax.text(0.98, 0.95, summary, transform=ax.transAxes, ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()


**What the result shows.** The two agree to the precision of the machine. $\varrho_{h}$ is 14954.2 from the whitened template and from `sigma` of `pycbc`. The largest difference between the two $\Phi(t_{c})$ is $6\times10^{-14}$. At the event both give 16.52, the value of lab H06. In the plot the dashed curve lies on the thick one at every time.

The standard matched filter is the correlation of the whitened data with the whitened template. `pycbc` divides by $S_{n}$ once, in the product. Lecture 14 divides each factor by $\sqrt{S_{n}}$. It is the same operation: whitening is part of the filter.


## Part B. Your turn: the AR model, two whitenings and a glitch

Part A showed how the data are whitened. In Part B we choose the order of the model, the whitening and the glitch. These are the functions we have defined for you. Nine are functions of Part A and three are new. Call them with keywords, as in Part A.

- `load_strain(path)` reads the strain, the times and the sampling frequency from the file `path`: `H1_FILE` or `L1_FILE`.
- `plot_series(times, x, ax=None, t_event=GPS_GW150914, window=None, ylabel='strain', color=None, label=None)` plots a series against the time from the event.
- `estimate_psd(x, fs, seglen=4.0)` returns the frequencies and $S_{n}$ with Welch's method.
- `plot_psd(freqs, psd, ax=None, label=None, color=None, lw=1.0, xlim=(10, 2000))` plots $\sqrt{S_{n}}$ on log axes. Call it again with the same `ax` to add a curve.
- `whiten_frequency(x, fs, spectrum=None, alpha=0.125)` returns the data whitened in frequency, with the pair (frequencies, $S_{n}$) of `estimate_psd`.
- `crop_edges(x, fs, start=2.0, stop=2.0)` returns a series without its first and last seconds.
- `fit_ar(x, order=512)` returns the coefficients $a_{k}$ and $\sigma$ of the AR model, with Burg's method.
- `ar_psd(a, sigma, fs, freqs)` returns $S_{n}$ of the AR model.
- `whiten_ar(x, a, sigma)` returns the data whitened in time with the prediction error filter. The first $p$ samples are not valid.
- `add_glitch(x, times, t0, f0=150.0, q=3.0, amplitude=1e-20)` returns the data with a simulated glitch: a sine-Gaussian at the time `t0`, the frequency `f0` and the peak strain `amplitude`.
- `time_frequency(x, times, fs, t_center, half=0.5, fmax=500.0)` returns the times, the frequencies and the logarithm of the power of the time–frequency map around `t_center`.
- `plot_time_frequency(t, f, power, ax=None, title=None)` plots that map.

`bandpass(x, fs, band=(35.0, 350.0))` of the course is also available.

**What we have from Part A.** `strain`, `times`, `fs`, `dt`, `freqs`, `psd`, `white`, `t_white`, `freqs_w`, `psd_w` and `models`.

**The glitch.** It is simulated: a sine-Gaussian, a model of a transient of the instrument. It is added to the strain of H1 and not to a simulated noise. The three new functions are defined in the next cell.

In [ ]:
# the three functions of Part B
def add_glitch(x, times, t0, f0=150.0, q=3.0, amplitude=1e-20):
    """Add a simulated glitch to the data: a sine-Gaussian, a model of a transient of the instrument.

    Args:
        x: the data, a one-dimensional array.
        times: the GPS time of each sample (s).
        t0: the time of the glitch, in the same units as times (s).
        f0: the frequency of the glitch (Hz, default 150).
        q: the number of cycles that the glitch lasts, about (default 3).
        amplitude: the peak strain of the glitch (default 1e-20).

    Returns:
        glitched: the data with the glitch added, as long as x.
    """
    # the duration of the Gaussian envelope
    tau = q / (2 * np.pi * f0)
    glitch = amplitude * np.exp(-((times - t0) / tau) ** 2) * np.cos(2 * np.pi * f0 * (times - t0))
    glitched = x + glitch
    return glitched


def time_frequency(x, times, fs, t_center, half=0.5, fmax=500.0):
    """Compute the time-frequency map of the data around a time.

    The map is the squared modulus of short Fourier transforms: a Hann window of 1/16 s,
    moved along the data in steps of 1/256 s.

    Args:
        x: the data, a one-dimensional array.
        times: the GPS time of each sample (s).
        fs: the sampling frequency (Hz).
        t_center: the time at the center of the map, in the same units as times (s).
        half: the half-width of the map (s, default 0.5).
        fmax: the highest frequency of the map (Hz, default 500).

    Returns:
        t: the times of the columns, from t_center (s).
        f: the frequencies of the rows, from 20 Hz to fmax (Hz).
        power: the base 10 logarithm of the power, an array of shape (len(f), len(t)).
    """
    # short Fourier transforms of 1/16 s, each one 1/256 s after the one before
    n_window = int(fs / 16)
    f_all, t_all, spec = spectrogram(x, fs=fs, window='hann', nperseg=n_window,
                                     noverlap=n_window - n_window // 16)
    # the times of the columns are counted from the start of the data
    t_all = t_all + times[0] - t_center
    in_time = np.abs(t_all) < half
    in_band = (f_all >= 20.0) & (f_all <= fmax)
    t = t_all[in_time]
    f = f_all[in_band]
    power = np.log10(spec[np.ix_(in_band, in_time)])
    return t, f, power


def plot_time_frequency(t, f, power, ax=None, title=None):
    """Plot a time-frequency map, with the color scale set from its own values.

    Args:
        t: the times of the columns (s).
        f: the frequencies of the rows (Hz).
        power: the base 10 logarithm of the power, from time_frequency.
        ax: the axes to draw on (default None: the current axes).
        title: the title of the plot (default None).

    Returns:
        ax: the axes that were drawn on.
    """
    if ax is None:
        ax = plt.gca()
    # the colors go from the 5th to the 99.9th percentile of this map
    mesh = ax.pcolormesh(t, f, power, cmap='magma', shading='auto',
                         vmin=np.percentile(power, 5), vmax=np.percentile(power, 99.9))
    plt.colorbar(mesh, ax=ax, label=r'$\log_{10}$ of the power')
    ax.set_xlabel('time from the center (s)')
    ax.set_ylabel('frequency (Hz)')
    if title is not None:
        ax.set_title(title, loc='left')
    return ax

### B1. The order of the AR model

Fit the AR model of H1 with four orders, for example 32, 128, 512 and 2048. Plot the four spectra over the Welch estimate and print $\sigma$ for each. Say what happens to the lines as the order grows.

*Hint:* `a, sigma = fit_ar(strain, order=...)`, then `ar_psd(a, sigma, fs=fs, freqs=freqs)` and `plot_psd` with the same `ax`.

In [ ]:
# your code here


> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B2. Two whitenings of the same data

Whiten H1 in frequency with the Welch PSD and in time with the AR model of the order you choose. Plot the PSD after each whitening on one axis and print the standard deviation of each whitened series. Which one is flatter?

*Hint:* `whiten_frequency` and `whiten_ar` both return a series as long as the data. Drop the edges with `crop_edges`: 2 s at each end for the first, the first $p$ samples (`start=p / fs`, `stop=0.0`) for the second. Then `estimate_psd` and `plot_psd`.

In [ ]:
# your code here


> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B3. A glitch before and after the whitening

Add a glitch to H1 at a time, a frequency and an amplitude of your choice. Start with an amplitude of $10^{-20}$. Whiten the data with the glitch in them. Make the time–frequency map around the glitch before and after the whitening. In which of the two is the glitch visible? Write one sentence that says why.

*Hint:* `add_glitch(strain, times, t0, f0=..., amplitude=...)`, then `estimate_psd`, `whiten_frequency` and `crop_edges`. The map is `time_frequency(x, times, fs=fs, t_center=t0)` for the raw data with `times` and for the whitened data with `t_white`. Draw both with `plot_time_frequency`.

In [ ]:
# your code here


> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B4. Another detector or another segment length (optional)

Repeat B2 on the Livingston data with `load_strain(L1_FILE)`. Or keep H1 and change `seglen` in `estimate_psd`. One thing changes: say which.

*Hint:* the same calls as in A2 and A3, with the other strain or the other `seglen`.

In [ ]:
# your code here


> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Software

`numpy` and `scipy.signal` for the PSD, the filters and the maps; `gwcourse_signal` to load the data and for the band-pass. The data are the 32 s of H1 and of L1 provided in `labs/data/`. The function `fit_ar` is Burg's recursion, the estimator of the AR model used for the whitening of detector data (Makhoul 1975).

The glitch of Part B is **simulated**. It is a model: the cached data contain no glitch near the event.

**References**

- Welch (1967). *The use of fast Fourier transform for the estimation of power spectra: A method based on time averaging over short, modified periodograms*.
- Makhoul (1975). *Linear prediction: A tutorial review*.
- LIGO Scientific Collaboration and Virgo Collaboration (2016). *Observation of Gravitational Waves from a Binary Black Hole Merger*.